In [1]:
BASE = "C:/Users/HP/Downloads/DS/da9/walmart-demand-forecast"

In [2]:
import pandas as pd
import duckdb

PROCESSED = f"{BASE}/data/processed"
results_df = pd.read_csv(f"{PROCESSED}/nhanh_a_full_results.csv")

# 1. Phân phối MAPE - có bị lệch bởi 1 nhóm nhỏ MAPE cực cao không?
print("=== PHÂN PHỐI MAPE (percentile) ===")
print(results_df['mape'].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]))

# 2. Bao nhiêu % có MAPE cực đoan (>100%, tức sai hơn cả giá trị thật)
extreme = results_df[results_df['mape'] > 100]
print(f"\nSố cặp có MAPE > 100%: {len(extreme)} ({len(extreme)/len(results_df)*100:.1f}%)")
print(f"Số cặp có MAPE > 50%: {(results_df['mape']>50).sum()} ({(results_df['mape']>50).mean()*100:.1f}%)")

# 3. Join với avg_sales để kiểm tra giả thuyết: MAPE cao có tương quan với
#    doanh số nhỏ không (giống pattern đã thấy ở Case 2)
con = duckdb.connect()
avg_sales = con.execute(f"""
    SELECT Store, Dept, AVG(Weekly_Sales) AS avg_sales, STDDEV(Weekly_Sales) AS std_sales
    FROM read_parquet('{PROCESSED}/walmart_clean.parquet')
    GROUP BY Store, Dept
""").df()

merged = results_df.merge(avg_sales, on=['Store','Dept'])
merged['cv'] = merged['std_sales'] / merged['avg_sales']  # coefficient of variation - đo độ biến động tương đối

# Chia theo quartile doanh số, xem MAPE trung bình mỗi nhóm
merged['sales_quartile'] = pd.qcut(merged['avg_sales'], 4, labels=['Q1_nhỏ_nhất','Q2','Q3','Q4_lớn_nhất'])
print("\n=== MAPE TRUNG BÌNH THEO QUARTILE DOANH SỐ ===")
print(merged.groupby('sales_quartile', observed=True).agg(
    mape_mean=('mape','mean'), mape_median=('mape','median'),
    beats_naive_rate=('beats_naive','mean'), n=('mape','count')
))

# Tương quan CV (độ biến động) với MAPE - giả thuyết: chuỗi biến động mạnh khó dự báo hơn
print(f"\nTương quan giữa CV (độ biến động) và MAPE: {merged['cv'].corr(merged['mape']):.3f}")

merged.to_csv(f"{PROCESSED}/nhanh_a_full_results_enriched.csv", index=False)

=== PHÂN PHỐI MAPE (percentile) ===
count     2847.000000
mean        31.764408
std        266.446310
min          0.758591
10%          4.254474
25%          7.111279
50%         12.613322
75%         22.416182
90%         42.284268
95%         68.578877
99%        235.354140
max      10969.538519
Name: mape, dtype: float64

Số cặp có MAPE > 100%: 104 (3.7%)
Số cặp có MAPE > 50%: 218 (7.7%)

=== MAPE TRUNG BÌNH THEO QUARTILE DOANH SỐ ===
                mape_mean  mape_median  beats_naive_rate    n
sales_quartile                                               
Q1_nhỏ_nhất     66.216914    25.347621          0.668539  712
Q2              34.007243    14.288137          0.563202  712
Q3              18.009272     9.770478          0.547117  711
Q4_lớn_nhất      8.804883     6.574414          0.571629  712

Tương quan giữa CV (độ biến động) và MAPE: 0.152


In [3]:
# Xem top 10 case MAPE cực đoan nhất - kiểm tra actual có gần 0 không
top_extreme = merged.sort_values('mape', ascending=False).head(10)
print(top_extreme[['Store','Dept','mape','avg_sales','std_sales','beats_naive']].to_string())

# Đếm bao nhiêu case actual test có giá trị rất nhỏ (gần 0) - nguyên nhân MAPE bùng nổ
con = duckdb.connect()
low_sales_check = con.execute(f"""
    SELECT Store, Dept, Date, Weekly_Sales
    FROM read_parquet('{PROCESSED}/walmart_clean.parquet')
    WHERE (Store, Dept) IN ({','.join([f"({r.Store},{r.Dept})" for _,r in top_extreme.head(5).iterrows()])})
      AND Date >= '2012-09-01'
    ORDER BY Store, Dept, Date
""").df()
print(low_sales_check.to_string())

      Store  Dept          mape     avg_sales    std_sales  beats_naive
2325     12    56  10969.538519   1000.415315  1044.019423         True
1396     35    16   8431.109435   7441.906364  8451.212465        False
1999     30    17   1525.922412    153.277833   109.579096         True
813      10    59   1312.277421   2006.511119  5481.527708        False
464       7    35   1017.708590    403.049021   238.269079         True
1780     42    31    920.438343     56.049219   116.690908        False
1433     37    56    914.907012    352.670862   361.621870         True
637      35    59    640.838766    830.778529  2864.769497         True
2227     24    54    636.294670     82.282426    90.803634         True
2657      8     3    565.125996  10683.305105  8098.106815        False
    Store  Dept       Date  Weekly_Sales
0       7    35 2012-09-07         19.06
1       7    35 2012-09-14        -17.47
2       7    35 2012-09-21         91.94
3       7    35 2012-09-28         43.47
4  

In [4]:
#sửa
import pandas as pd

results_df = pd.read_csv(f"{PROCESSED}/nhanh_a_full_results_enriched.csv")

# Loại các case actual quá nhỏ khỏi thống kê MAPE tổng hợp (không loại khỏi model,
# chỉ loại khỏi cách BÁO CÁO trung bình - vẫn giữ nguyên forecast cho các case này)
print("=== SO SÁNH CÁC CÁCH BÁO CÁO ĐỘ CHÍNH XÁC ===")
print(f"Mean MAPE (bị outlier chi phối - KHÔNG dùng để báo cáo): {results_df['mape'].mean():.2f}%")
print(f"Median MAPE (robust hơn, nên dùng): {results_df['mape'].median():.2f}%")

# Loại các case actual trung bình quá nhỏ (< $100/tuần) - nhóm này MAPE về bản chất
# không phải thước đo phù hợp, nên báo cáo riêng bằng USD tuyệt đối, không phần trăm
small_volume = results_df[results_df['avg_sales'] < 100]
main_volume = results_df[results_df['avg_sales'] >= 100]

print(f"\nSố cặp doanh số < $100/tuần (loại khỏi MAPE tổng hợp): {len(small_volume)} ({len(small_volume)/len(results_df)*100:.1f}%)")
print(f"MAPE trung vị của nhóm chính (>= $100/tuần): {main_volume['mape'].median():.2f}%")
print(f"Thắng naive của nhóm chính: {main_volume['beats_naive'].mean()*100:.1f}%")

=== SO SÁNH CÁC CÁCH BÁO CÁO ĐỘ CHÍNH XÁC ===
Mean MAPE (bị outlier chi phối - KHÔNG dùng để báo cáo): 31.76%
Median MAPE (robust hơn, nên dùng): 12.61%

Số cặp doanh số < $100/tuần (loại khỏi MAPE tổng hợp): 90 (3.2%)
MAPE trung vị của nhóm chính (>= $100/tuần): 12.24%
Thắng naive của nhóm chính: 58.3%
